# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os

os.listdir("/content")

['.config', 'content_refresh_anonymized.csv', 'sample_data']

In [11]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 53)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


In [12]:
print("Full dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nTrend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

Full dataset shape: (30000, 53)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', 'is_initial_refresh_candidate']

Tre

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [13]:
print("ML task type: Ranking")
print("Total pages:", len(df))
print("Pages labeled as declining:", (df["trend_direction"] == "down").sum())
print("Pages not labeled as declining:", (df["trend_direction"] != "down").sum())

ML task type: Ranking
Total pages: 30000
Pages labeled as declining: 16262
Pages not labeled as declining: 13738


## 2. Target or proxy

My target is down_trend_proxy, derived from the existing trend_direction column. Pages labeled down receive a value of 1, while all other pages receive a value of 0. This proxy helps identify pages whose search performance is currently declining.

The target is suitable for prioritizing pages for human review. However, it represents an existing trend label rather than a prediction of future decline. I will avoid using trend_direction or any equivalent target-derived information as model input to prevent data leakage.

In [14]:
df["down_trend_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target column created successfully.")
print(df["down_trend_proxy"].value_counts())

Target column created successfully.
down_trend_proxy
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

My primary success metric is Precision@50. It measures how many of the top 50 pages selected by the ML ranking are actually labeled as declining. This metric is suitable because content editors have limited time and need to review the highest-priority pages first.

I will compare the ML ranking against a simple rule-based baseline to check whether ML improves the quality of the recommendations. ROC-AUC can be used as a secondary metric if the system produces prediction scores.

The goal is to help editors prioritize pages for review, not to automatically update content or claim that updating a page will improve its search performance.

In [15]:
k = 50

positive_pages = int(df["down_trend_proxy"].sum())
total_pages = len(df)

print("Total pages:", total_pages)
print("Pages labeled as declining:", positive_pages)
print("Pages available for Precision@50:", min(k, total_pages))
print("Positive-label percentage:", round(positive_pages / total_pages * 100, 2), "%")

Total pages: 30000
Pages labeled as declining: 16262
Pages available for Precision@50: 50
Positive-label percentage: 54.21 %


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content page identified by content_id. Each row represents a page and contains its search performance, traffic metrics, content freshness, and trend information.

My dataset contains 30,000 rows and 53 original columns. I created an additional column, down_trend_proxy, where 1 indicates a page labeled as declining and 0 indicates a page not labeled as declining.

The ML system will rank individual content pages so that editors can prioritize pages for human review

In [16]:
# Show a real dataframe slice for the Content Refresh / Decline Risk lane

columns_to_show = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "down_trend_proxy"
]

display(df[columns_to_show].head(10))

,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction,down_trend_proxy
0,content_304f48230142,3803,29,0.76,10.6,187,20,down,1
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445,25,down,1
2,content_9aa793d4d895,12581,11,0.09,36.5,141,20,down,1
3,content_331d6c4de07b,11751,58,0.49,6.2,463,22,stable,0
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263,14,down,1
5,content_d4084a4bc775,3970,1,0.03,8.5,147,20,down,1
6,content_9a34b442b552,20,0,0.00,7.0,90,20,down,1
7,content_a63219c6e95a,1724,1,0.06,21.2,445,22,stable,0
8,content_5e6c160719bc,32574,29,0.09,46.0,90,20,down,1
9,content_c27558df2b0c,1240,2,0.16,4.9,257,104,down,1


## 5. Why ML beats a fixed rule here
A fixed rule might prioritize pages only when impressions drop below a specific threshold. However, declining search performance can be associated with several factors, including impressions, clicks, CTR, average position, content age, and days since the last update. Different combinations of these signals may indicate different levels of risk.

Machine learning can combine multiple signals to produce a ranking of pages for human review instead of relying on a single threshold. I will compare the ML ranking with a simple rule-based baseline using Precision@50 to determine whether ML provides additional value. If ML does not outperform the baseline, the simpler rule may be preferable.

The final decision to update or investigate a page will remain with the human editor.

In [17]:
features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

display(df[features].describe().round(2))

print("\nDeclining vs. non-declining pages:")
display(
    df.groupby("down_trend_proxy")[features]
      .mean()
      .round(2)
)

,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
count,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00
mean,5200.37,16.10,0.51,16.34,256.17,46.10
std,16838.02,75.08,3.28,15.22,132.71,42.08
min,1.00,0.00,0.00,0.00,90.00,1.00
25%,81.00,0.00,0.00,6.20,132.00,20.00
50%,731.00,1.00,0.07,10.80,236.00,20.00
75%,3615.25,7.00,0.29,22.30,333.00,104.00
max,517715.00,4178.00,100.00,245.00,564.00,373.00



Declining vs. non-declining pages:


,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
down_trend_proxy,,,,,,
0,5533.31,19.34,0.73,16.82,279.83,42.37
1,4919.10,13.36,0.32,15.94,236.18,49.25


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.